In [ ]:
%%capture
!pip install accelerate  -U # and restart the kernel

In [ ]:
%%capture
!pip install seqeval
!pip install  datasets

In [ ]:
import os, sys
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
#gpu="0,1,2,3"
gpu="0"
os.environ["CUDA_VISIBLE_DEVICES"]=gpu
import numpy as np
import seqeval.metrics
from seqeval.scheme import IOB2
import transformers
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
from transformers import DataCollatorForTokenClassification

from datasets import load_dataset, load_metric, Dataset, DatasetDict

from sklearn.model_selection import train_test_split

In [ ]:
MODEL = 'roberta' #or roberta
if MODEL == 'bert':
    print("Using BERT model")
    model_checkpoint = "bert-base-multilingual-cased"
    batch_size = 128
    learning_rate = 5e-5
    weight_decay = 0.0001
    epochs = 8
    warmup_steps = 3000
    seed = 1
elif MODEL == 'roberta':
    print("Using XLM-RoBERTa model")
    model_checkpoint = "xlm-roberta-base" #"xlm-roberta-large"
    batch_size = 64
    learning_rate = 1e-5
    weight_decay = 0.001
    epochs = 10
    warmup_steps = 800
    seed = 1
else:
    print("Usage example:\n python run_finetune_kaznerd.py model (bert|roberta)\n"
            "e.g.: python run_finetune_kaznerd.py roberta")
    exit()



Using XLM-RoBERTa model


In [ ]:
task = "ner"

#print(transformers.__version__)

def tokenize_and_align_labels(examples, tokenizer, task, label_all_tokens=False):
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True,
                        is_split_into_words=True)

    labels = []
    for i, label in enumerate(examples[f"{task}_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            # Special tokens have a word id that is None. We set the label to -100 so they are
            # automatically ignored in the loss function.
            if word_idx is None:
                label_ids.append(-100)
            # We set the label for the first token of each word.
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
                # For the other tokens in a word, we set the label to either the current label or
                # -100, depending on the label_all_tokens flag.
            else:
                label_ids.append(label[word_idx] if label_all_tokens else -100)
            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                            for prediction, label in zip(predictions, labels)]
    true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100]
                        for prediction, label in zip(predictions, labels)]

    #computes micro average
    results = metric.compute(predictions=true_predictions, references=true_labels,
                scheme="IOB2", mode="strict")
    return {"precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"]}



In [ ]:
def read_conll2003mod(filepath: str = "something.conll2003mod-formatted.txt",
                      labels_available=True, doc_ids_available=False):
    ids, sequences, doc_ids = [], [], []

    with open(filepath, "r", encoding="utf-8") as rf:

        for line in rf:
            line = line.strip()

            # skipping empty lines
            if line:
                # reading the sentence ID
                if line.startswith("#"):
                    splitted = line.strip("#").strip().split(" ")
                    ids.append(int(splitted[0]))
                    if doc_ids_available:
                        doc_ids.append(int(splitted[1]))
                    sequences.append([])

                # reading the token-tag pair
                else:
                    token = line.split("\t")[0]
                    if labels_available:
                        tag = line.split("\t")[3]
                    else:
                        tag = -1
                    sequences[-1].append((token, tag))

    assert len(ids) == len(sequences)

    if doc_ids_available:
        return ids, sequences, doc_ids

    return ids, sequences

# def read_conll_file(file_path):
#     with open(file_path, "r") as f:
#         content = f.read().strip()
#         sentences = content.split("\n\n")
#         data = []
#         for sentence in sentences:
#             tokens = sentence.split("\n")
#             token_data = []
#             for token in tokens:
#                 token_data.append(token.split())
#             data.append(token_data)
#     return data


# train_data = read_conll_file("/content/drive/MyDrive/The_Cramer/AkylAI/NER/Codes/KyrgyzNER_TRAIN.jsonl.txt")
# validation_data = read_conll_file("/kaggle/input/conll2003-dataset/conll2003/eng.testa")
# test_data = read_conll_file("/kaggle/input/conll2003-dataset/conll2003/eng.testb")

data_id, data = read_conll2003mod('KyrgyzNER_TRAIN.jsonl.txt')

test_id, test = read_conll2003mod('KyrgyzNER_TEST.jsonl.gold.txt')

train, valid = train_test_split(data, random_state=42, test_size=0.2)

print(len(train), len(valid), len(test))

label_list_train = set([token_data[1] for sentence in data for token_data in sentence])
label_list_test = set([token_data[1] for sentence in test for token_data in sentence])
label_list = list(label_list_train | label_list_test)

label_map = {label: i for i, label in enumerate(label_list)}

print(len(label_list))


5626 1407 3867
51


In [ ]:
label_list

['B-NATIONAL',
 'I-PLANT',
 'I-ORGANISATION',
 'B-ORGANISATION',
 'B-MEDIA',
 'I-ARTIFACT',
 'B-AWARD',
 'B-UNKNOWN',
 'I-LOCATION',
 'B-PERSON',
 'I-LEGAL',
 'B-BUSINESS',
 'B-ACRONYM',
 'I-PERIOD',
 'B-INSTITUTION',
 'I-MEASURE',
 'B-CREATION',
 'I-ACRONYM',
 'I-AWARD',
 'I-WEBSITE',
 'B-PERIOD',
 'I-PERSON',
 'I-PERSON_TYPE',
 'B-SUBSTANCE',
 'O',
 'B-PLANT',
 'I-INSTITUTION',
 'I-SUBSTANCE',
 'I-INSTALLATION',
 'B-CONCEPT',
 'B-TITLE',
 'I-EVENT',
 'B-ARTIFACT',
 'B-MEASURE',
 'B-LOCATION',
 'I-BUSINESS',
 'B-ANIMAL',
 'B-PERSON_TYPE',
 'B-INSTALLATION',
 'I-TITLE',
 'B-IDENTIFIER',
 'I-IDENTIFIER',
 'B-LEGAL',
 'I-MEDIA',
 'I-CONCEPT',
 'I-UNKNOWN',
 'B-EVENT',
 'B-WEBSITE',
 'I-NATIONAL',
 'I-CREATION',
 'I-ANIMAL']

In [ ]:
label_map

{'B-NATIONAL': 0,
 'I-PLANT': 1,
 'I-ORGANISATION': 2,
 'B-ORGANISATION': 3,
 'B-MEDIA': 4,
 'I-ARTIFACT': 5,
 'B-AWARD': 6,
 'B-UNKNOWN': 7,
 'I-LOCATION': 8,
 'B-PERSON': 9,
 'I-LEGAL': 10,
 'B-BUSINESS': 11,
 'B-ACRONYM': 12,
 'I-PERIOD': 13,
 'B-INSTITUTION': 14,
 'I-MEASURE': 15,
 'B-CREATION': 16,
 'I-ACRONYM': 17,
 'I-AWARD': 18,
 'I-WEBSITE': 19,
 'B-PERIOD': 20,
 'I-PERSON': 21,
 'I-PERSON_TYPE': 22,
 'B-SUBSTANCE': 23,
 'O': 24,
 'B-PLANT': 25,
 'I-INSTITUTION': 26,
 'I-SUBSTANCE': 27,
 'I-INSTALLATION': 28,
 'B-CONCEPT': 29,
 'B-TITLE': 30,
 'I-EVENT': 31,
 'B-ARTIFACT': 32,
 'B-MEASURE': 33,
 'B-LOCATION': 34,
 'I-BUSINESS': 35,
 'B-ANIMAL': 36,
 'B-PERSON_TYPE': 37,
 'B-INSTALLATION': 38,
 'I-TITLE': 39,
 'B-IDENTIFIER': 40,
 'I-IDENTIFIER': 41,
 'B-LEGAL': 42,
 'I-MEDIA': 43,
 'I-CONCEPT': 44,
 'I-UNKNOWN': 45,
 'B-EVENT': 46,
 'B-WEBSITE': 47,
 'I-NATIONAL': 48,
 'I-CREATION': 49,
 'I-ANIMAL': 50}

In [ ]:
def convert_to_dataset(data):
    formatted_data = {"tokens": [], "ner_tags": []}

    for sentence in data:

        tokens = [token_data[0] for token_data in sentence]
        ner_tags = [label_map[token_data[1]]  for token_data in sentence]
        # ner_tags = [token_data[1]  for token_data in sentence]

        formatted_data["tokens"].append(tokens)
        formatted_data["ner_tags"].append(ner_tags)

    return Dataset.from_dict(formatted_data)


train_dataset = convert_to_dataset(train)
validation_dataset = convert_to_dataset(valid)
test_dataset = convert_to_dataset(test)


datasets = DatasetDict({
    "train": train_dataset,
    "validation": validation_dataset,
    "test": test_dataset,
})


In [ ]:

# datasets = load_dataset("kaznerd.py")
# label_list = datasets["train"].features[f"{task}_tags"].feature.names


tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

assert isinstance(tokenizer, transformers.PreTrainedTokenizerFast)
data_collator = DataCollatorForTokenClassification(tokenizer)

metric = load_metric("seqeval")

model = AutoModelForTokenClassification.from_pretrained(model_checkpoint,
            num_labels=len(label_list))

model_name = model_checkpoint.split("/")[-1]

args = TrainingArguments(f"{model_name}-kyrgyzNER",
                            overwrite_output_dir=True,
                            evaluation_strategy="epoch",
                            per_device_train_batch_size=batch_size,
                            per_device_eval_batch_size=batch_size,
                            learning_rate=learning_rate,
                            num_train_epochs=epochs,
                            warmup_steps=warmup_steps,
                            weight_decay=weight_decay,
                            save_strategy="no",
                            seed=seed,
                            push_to_hub=False)

tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True,
        fn_kwargs={"tokenizer":tokenizer,"task":task})

trainer = Trainer(model, args,
                  data_collator=data_collator,
                  train_dataset=tokenized_datasets["train"],
                  eval_dataset=tokenized_datasets["validation"],
                  tokenizer=tokenizer,
                  compute_metrics=compute_metrics)

trainer.train()
#trainer.evaluate()


/usr/local/lib/python3.10/dist-packages/datasets/load.py:753: FutureWarning: The repository for seqeval contains custom code which must be executed to correctly load the metric. You can inspect the repository content at https://raw.githubusercontent.com/huggingface/datasets/2.17.0/metrics/seqeval/seqeval.py
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this metric from the next major release of `datasets`.
  warnings.warn(
Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/5626 [00:00<?, ? examples/s]

Map:   0%|          | 0/1407 [00:00<?, ? examples/s]

Map:   0%|          | 0/3867 [00:00<?, ? examples/s]

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,3.813549,0.000109,0.000883,0.000195,0.000164
2,No log,1.386714,0.000000,0.000000,0.000000,0.716199
3,No log,0.927596,0.168399,0.035746,0.058973,0.767809
4,No log,0.646655,0.446975,0.277140,0.342141,0.841999
5,No log,0.520039,0.639423,0.528244,0.578540,0.879176
6,1.647000,0.438302,0.671171,0.591792,0.628987,0.892734
7,1.647000,0.384718,0.672350,0.643866,0.657800,0.902794
8,1.647000,0.358626,0.685688,0.657546,0.671322,0.906129
9,1.647000,0.342169,0.678556,0.671668,0.675094,0.907003
10,1.647000,0.322937,0.704206,0.687114,0.695555,0.911924


/usr/local/lib/python3.10/dist-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=880, training_loss=1.0864589170976118, metrics={'train_runtime': 669.0012, 'train_samples_per_second': 84.096, 'train_steps_per_second': 1.315, 'total_flos': 1786432864841460.0, 'train_loss': 1.0864589170976118, 'epoch': 10.0})

In [ ]:

#################################################################################################
#Evaluate validation set
print("#"*100)
predictions, labels, _ = trainer.predict(tokenized_datasets["validation"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                    for prediction, label in zip(predictions, labels)]
true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100]
               for prediction, label in zip(predictions, labels)]

results = metric.compute(predictions=true_predictions, references=true_labels, scheme="IOB2",
            mode="strict")
print("\nValidation: Overall F1", results["overall_f1"])
print("Validation: Total number of sentences:",len(true_labels))
print("Validation: Total number of tokens:", sum([len(sent) for sent in true_labels]))
print("Validation: seqeval based results")
print(seqeval.metrics.classification_report(true_labels, true_predictions, digits=4, mode='strict',
    scheme=IOB2))


####################################################################################################



Validation: Overall F1 0.6955550591914229
Validation: Total number of sentences: 1407
Validation: Total number of tokens: 18291
Validation: seqeval based results
              precision    recall  f1-score   support

     ACRONYM     0.0000    0.0000    0.0000         8
    ARTIFACT     0.0000    0.0000    0.0000        26
       AWARD     0.0000    0.0000    0.0000         2
    BUSINESS     0.3971    0.5745    0.4696        47
     CONCEPT     0.0000    0.0000    0.0000        18
    CREATION     0.0000    0.0000    0.0000         6
       EVENT     0.0000    0.0000    0.0000        26
  IDENTIFIER     0.0000    0.0000    0.0000         1
INSTALLATION     0.0000    0.0000    0.0000         9
 INSTITUTION     0.5355    0.7295    0.6176       207
       LEGAL     0.0000    0.0000    0.0000        29
    LOCATION     0.6910    0.7470    0.7179       494
     MEASURE     0.7953    0.8382    0.8162       482
       MEDIA     0.8125    0.6964    0.7500        56
    NATIONAL     1.0000   

In [ ]:

#################################################################################################
#Evaluate test set
print("#"*100)
predictions, labels, _ = trainer.predict(tokenized_datasets["test"])
predictions = np.argmax(predictions, axis=2)

# Remove ignored index (special tokens)
true_predictions = [[label_list[p] for (p, l) in zip(prediction, label) if l != -100]
                    for prediction, label in zip(predictions, labels)]
true_labels = [[label_list[l] for (p, l) in zip(prediction, label) if l != -100]
               for prediction, label in zip(predictions, labels)]

results = metric.compute(predictions=true_predictions, references=true_labels, scheme="IOB2",
            mode="strict")
print("\nTest: Overall F1", results["overall_f1"])
print("Test: Total number of sentences:",len(true_labels))
print("Test: Total number of tokens:", sum([len(sent) for sent in true_labels]))
print("Test: seqeval based results")
print(seqeval.metrics.classification_report(true_labels, true_predictions, digits=4, mode='strict',
    scheme=IOB2))
print("#"*100)

####################################################################################################



Test: Overall F1 0.6860608191246265
Test: Total number of sentences: 3867
Test: Total number of tokens: 51118
Test: seqeval based results
              precision    recall  f1-score   support

     ACRONYM     0.0000    0.0000    0.0000        14
      ANIMAL     0.0000    0.0000    0.0000         3
    ARTIFACT     0.0000    0.0000    0.0000        36
       AWARD     0.0000    0.0000    0.0000         7
    BUSINESS     0.3861    0.5132    0.4407        76
     CONCEPT     0.0000    0.0000    0.0000        99
    CREATION     0.2000    0.0417    0.0690        48
       EVENT     0.0000    0.0000    0.0000        49
  IDENTIFIER     0.0000    0.0000    0.0000         8
INSTALLATION     0.0000    0.0000    0.0000        14
 INSTITUTION     0.6206    0.6108    0.6157       758
       LEGAL     0.0000    0.0000    0.0000        54
    LOCATION     0.6586    0.7938    0.7199      1033
     MEASURE     0.8681    0.8286    0.8479      1231
       MEDIA     0.7438    0.8036    0.7725       

In [ ]:

label_map_id2label = {j:i for i, j in label_map.items() }
label_map_id2label

{0: 'B-CONCEPT',
 1: 'B-WEBSITE',
 2: 'B-ARTIFACT',
 3: 'I-ORGANISATION',
 4: 'I-LEGAL',
 5: 'B-ACRONYM',
 6: 'B-BUSINESS',
 7: 'B-ANIMAL',
 8: 'I-INSTALLATION',
 9: 'I-MEDIA',
 10: 'I-ANIMAL',
 11: 'B-CREATION',
 12: 'I-UNKNOWN',
 13: 'I-SUBSTANCE',
 14: 'I-PERSON',
 15: 'B-MEASURE',
 16: 'I-AWARD',
 17: 'I-LOCATION',
 18: 'I-PERIOD',
 19: 'I-TITLE',
 20: 'B-MEDIA',
 21: 'B-AWARD',
 22: 'B-IDENTIFIER',
 23: 'B-PLANT',
 24: 'B-LEGAL',
 25: 'B-UNKNOWN',
 26: 'I-NATIONAL',
 27: 'I-IDENTIFIER',
 28: 'B-PERSON',
 29: 'B-PERSON_TYPE',
 30: 'B-INSTITUTION',
 31: 'I-ACRONYM',
 32: 'I-MEASURE',
 33: 'B-TITLE',
 34: 'O',
 35: 'I-INSTITUTION',
 36: 'I-PERSON_TYPE',
 37: 'I-CREATION',
 38: 'I-ARTIFACT',
 39: 'B-EVENT',
 40: 'I-EVENT',
 41: 'B-PERIOD',
 42: 'I-BUSINESS',
 43: 'B-ORGANISATION',
 44: 'I-PLANT',
 45: 'B-LOCATION',
 46: 'B-NATIONAL',
 47: 'B-INSTALLATION',
 48: 'B-SUBSTANCE',
 49: 'I-CONCEPT',
 50: 'I-WEBSITE'}

In [ ]:
# model.config.id2label UPDATE: Done

model.config.id2label.update(label_map_id2label)
model.config.id2label

{0: 'B-CONCEPT',
 1: 'B-WEBSITE',
 2: 'B-ARTIFACT',
 3: 'I-ORGANISATION',
 4: 'I-LEGAL',
 5: 'B-ACRONYM',
 6: 'B-BUSINESS',
 7: 'B-ANIMAL',
 8: 'I-INSTALLATION',
 9: 'I-MEDIA',
 10: 'I-ANIMAL',
 11: 'B-CREATION',
 12: 'I-UNKNOWN',
 13: 'I-SUBSTANCE',
 14: 'I-PERSON',
 15: 'B-MEASURE',
 16: 'I-AWARD',
 17: 'I-LOCATION',
 18: 'I-PERIOD',
 19: 'I-TITLE',
 20: 'B-MEDIA',
 21: 'B-AWARD',
 22: 'B-IDENTIFIER',
 23: 'B-PLANT',
 24: 'B-LEGAL',
 25: 'B-UNKNOWN',
 26: 'I-NATIONAL',
 27: 'I-IDENTIFIER',
 28: 'B-PERSON',
 29: 'B-PERSON_TYPE',
 30: 'B-INSTITUTION',
 31: 'I-ACRONYM',
 32: 'I-MEASURE',
 33: 'B-TITLE',
 34: 'O',
 35: 'I-INSTITUTION',
 36: 'I-PERSON_TYPE',
 37: 'I-CREATION',
 38: 'I-ARTIFACT',
 39: 'B-EVENT',
 40: 'I-EVENT',
 41: 'B-PERIOD',
 42: 'I-BUSINESS',
 43: 'B-ORGANISATION',
 44: 'I-PLANT',
 45: 'B-LOCATION',
 46: 'B-NATIONAL',
 47: 'B-INSTALLATION',
 48: 'B-SUBSTANCE',
 49: 'I-CONCEPT',
 50: 'I-WEBSITE'}

In [ ]:
model.config.label2id = {j:i for i, j in model.config.id2label.items()}
model.config.label2id

{'B-CONCEPT': 0,
 'B-WEBSITE': 1,
 'B-ARTIFACT': 2,
 'I-ORGANISATION': 3,
 'I-LEGAL': 4,
 'B-ACRONYM': 5,
 'B-BUSINESS': 6,
 'B-ANIMAL': 7,
 'I-INSTALLATION': 8,
 'I-MEDIA': 9,
 'I-ANIMAL': 10,
 'B-CREATION': 11,
 'I-UNKNOWN': 12,
 'I-SUBSTANCE': 13,
 'I-PERSON': 14,
 'B-MEASURE': 15,
 'I-AWARD': 16,
 'I-LOCATION': 17,
 'I-PERIOD': 18,
 'I-TITLE': 19,
 'B-MEDIA': 20,
 'B-AWARD': 21,
 'B-IDENTIFIER': 22,
 'B-PLANT': 23,
 'B-LEGAL': 24,
 'B-UNKNOWN': 25,
 'I-NATIONAL': 26,
 'I-IDENTIFIER': 27,
 'B-PERSON': 28,
 'B-PERSON_TYPE': 29,
 'B-INSTITUTION': 30,
 'I-ACRONYM': 31,
 'I-MEASURE': 32,
 'B-TITLE': 33,
 'O': 34,
 'I-INSTITUTION': 35,
 'I-PERSON_TYPE': 36,
 'I-CREATION': 37,
 'I-ARTIFACT': 38,
 'B-EVENT': 39,
 'I-EVENT': 40,
 'B-PERIOD': 41,
 'I-BUSINESS': 42,
 'B-ORGANISATION': 43,
 'I-PLANT': 44,
 'B-LOCATION': 45,
 'B-NATIONAL': 46,
 'B-INSTALLATION': 47,
 'B-SUBSTANCE': 48,
 'I-CONCEPT': 49,
 'I-WEBSITE': 50}